In [ ]:
#Train Random Forest Regressor & Tune Hyperparameters

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings, datetime
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
# Reload data and recreate the same split as June 22
df_c = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
df_c['year']  = df_c['Date'].dt.year
df_c['month'] = df_c['Date'].dt.month

FEATURES_RF = ['SST (°C)', 'pH Level']
TARGET_RF   = 'Species Observed'

rf_data = df_c[FEATURES_RF + [TARGET_RF]].dropna().copy()
X = rf_data[FEATURES_RF]
y = rf_data[TARGET_RF]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Target range: {y.min():.0f} – {y.max():.0f}  |  Mean: {y.mean():.2f}')

In [ ]:
#Step 1: Baseline Random Forest (Default Parameters)
# Train a baseline model first — no tuning, default depth
rf_base = RandomForestRegressor(n_estimators=100, random_state=42)
rf_base.fit(X_train, y_train)
yp_base = rf_base.predict(X_test)

r2_base  = r2_score(y_test, yp_base)
mae_base = mean_absolute_error(y_test, yp_base)

print('BASELINE RANDOM FOREST  (n_estimators=100, max_depth=None)')
print('='*60)
print(f'  R²  : {r2_base:.4f}   ({r2_base*100:.1f}% of test variance explained)')
print(f'  MAE : {mae_base:.4f}  (average prediction error in species)')
print()
print('Feature importances (baseline — unlimited depth):')
for feat, imp in zip(FEATURES_RF, rf_base.feature_importances_):
    bar = '█' * int(imp * 30)
    print(f'  {feat:<22}: {imp:.4f}  ({imp*100:.2f}%)  {bar}')

In [ ]:
#Step 2: Hyperparameter Tuning with GridSearchCV
# Grid of hyperparameters to search:
# n_estimators — number of trees in the forest
# max_depth    — max depth per tree (None = unlimited)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth':    [3, 5, None],
}

print('GRIDSEARCHCV — 3-FOLD CROSS VALIDATION')
print('='*55)
print(f'  Parameter grid : {param_grid}')
print(f'  Total combos   : {2*3} (2 estimators × 3 depths)')
print(f'  CV folds       : 3  (each combo evaluated 3 times)')
print(f'  Scoring        : R²')
print(f'  Total fits     : {2*3*3} = 36')
print()
print('Running GridSearchCV...')

gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
gs.fit(X_train, y_train)

print(f'Done.')
print()
print(f'Best parameters : {gs.best_params_}')
print(f'Best CV R²      : {gs.best_score_:.4f}')

In [ ]:
# Show all GridSearch results sorted by performance
print('ALL GRIDSEARCH RESULTS (sorted by mean CV R²):')
print('='*65)
print(f'  {"n_estimators":<15} {"max_depth":<12} {"mean CV R²":>12} {"std":>8}')
print('-'*50)

res_df = pd.DataFrame(gs.cv_results_)[[
    'param_n_estimators','param_max_depth','mean_test_score','std_test_score'
]].sort_values('mean_test_score', ascending=False).reset_index(drop=True)

for _, row in res_df.iterrows():
    flag = ' ← BEST' if (row['param_n_estimators']==100 and str(row['param_max_depth'])=='3') else ''
    print(f'  {str(row["param_n_estimators"]):<15} {str(row["param_max_depth"]):<12} '
          f'{row["mean_test_score"]:>12.4f} {row["std_test_score"]:>8.4f}{flag}')

print()
print('INTERPRETATION:')
print('  max_depth=3  consistently outperforms deeper trees (CV R²=0.3498).')
print('  max_depth=None (unlimited) performs worst — overfits training data.')
print('  Limiting depth forces the model to prioritize SST as primary split.')

In [ ]:
#Step 3: Evaluate Best Model on Held-Out Test Set
best_rf = gs.best_estimator_
yp_best = best_rf.predict(X_test)

r2_best  = r2_score(y_test, yp_best)
mae_best = mean_absolute_error(y_test, yp_best)

print('BEST MODEL TEST SET EVALUATION')
print('='*60)
print(f'  Best parameters : n_estimators=100, max_depth=3')
print(f'  CV R² (train)   : {gs.best_score_:.4f}')
print(f'  Test R²         : {r2_best:.4f}   ({r2_best*100:.1f}% of test variance explained)')
print(f'  Test MAE        : {mae_best:.4f}  (avg ±{mae_best:.1f} species error)')
print()
print('IMPROVEMENT OVER BASELINE:')
print(f'  R²  improvement : {r2_base:.4f} → {r2_best:.4f}  (+{r2_best-r2_base:.4f})')
print(f'  MAE improvement : {mae_base:.4f} → {mae_best:.4f}  (better by {mae_base-mae_best:.4f} species)')
print()
print('FEATURE IMPORTANCES (tuned model, max_depth=3):')
for feat, imp in zip(FEATURES_RF, best_rf.feature_importances_):
    bar = '█' * int(imp * 40)
    print(f'  {feat:<22}: {imp:.4f}  ({imp*100:.2f}%)  {bar}')


In [ ]:
#Step 4: Predicted vs Actual + Residual Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sns.set_style('whitegrid')

# ── Panel 1: Predicted vs Actual Scatter ──
ax1 = axes[0]
ax1.scatter(y_test, yp_best, alpha=0.5, color='steelblue', s=35,
            label='Predictions', edgecolors='none')
lims = [min(float(y_test.min()), yp_best.min()) - 5,
        max(float(y_test.max()), yp_best.max()) + 5]
ax1.plot(lims, lims, 'r--', linewidth=2.5, label='Perfect prediction (y = x)')
ax1.set_xlabel('Actual Species Observed', fontsize=11)
ax1.set_ylabel('Predicted Species Observed', fontsize=11)
ax1.set_title(f'Predicted vs Actual\nR² = {r2_best:.4f}  |  MAE = {mae_best:.2f}',
              fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# ── Panel 2: Residual Plot ──
ax2 = axes[1]
residuals = y_test.values - yp_best
ax2.scatter(yp_best, residuals, alpha=0.5, color='#e74c3c', s=35, edgecolors='none')
ax2.axhline(y=0,  color='black', linestyle='--', linewidth=2, label='Zero residual')
ax2.axhline(y=14.59, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='±1 std (14.59)')
ax2.axhline(y=-14.59, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.set_xlabel('Predicted Species', fontsize=11)
ax2.set_ylabel('Residuals (Actual − Predicted)', fontsize=11)
ax2.set_title(f'Residual Plot\nmean={residuals.mean():.2f}, std={residuals.std():.2f}',
              fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

# ── Panel 3: Feature Importance Bar ──
ax3 = axes[2]
importance_vals = best_rf.feature_importances_
colors_imp = ['#e74c3c' if i==0 else '#3498db' for i in range(len(FEATURES_RF))]
bars = ax3.barh(FEATURES_RF, importance_vals,
                color=colors_imp, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, importance_vals):
    ax3.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val*100:.2f}%', va='center', fontsize=11, fontweight='bold')
ax3.set_xlabel('Feature Importance', fontsize=11)
ax3.set_title('Feature Importances\n(max_depth=3)', fontweight='bold')
ax3.set_xlim(0, 1.1)
ax3.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax3.grid(True, alpha=0.3, axis='x')

plt.suptitle(f'June 23, 2026 — Random Forest Regressor Evaluation\n'
             f'Best params: max_depth=3, n_estimators=100',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week6_Day2_rf_evaluation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('RF evaluation chart saved ✅')

In [ ]:
#Step 5: Save Model Weights & Metadata
import joblib

# Save trained model weights
model_path = DIRS['processed']+'/rf_best_model.joblib'
joblib.dump(best_rf, model_path)

# Save all model metadata
rf_meta = {
    'model':          'RandomForestRegressor',
    'best_params':    gs.best_params_,
    'cv_r2':          round(float(gs.best_score_), 4),
    'test_r2':        round(r2_best, 4),
    'test_mae':       round(mae_best, 4),
    'baseline_r2':    round(r2_base, 4),
    'baseline_mae':   round(mae_base, 4),
    'feature_importances': {
        f: round(float(i), 4)
        for f, i in zip(FEATURES_RF, best_rf.feature_importances_)
    },
    'n_train':        int(len(X_train)),
    'n_test':         int(len(X_test)),
    'residuals': {
        'mean': round(float(residuals.mean()), 4),
        'std':  round(float(residuals.std()),  4),
        'min':  round(float(residuals.min()),  2),
        'max':  round(float(residuals.max()),  2),
    },
    'generated': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
}

with open(DIRS['metadata']+'/rf_model_meta.json', 'w') as f:
    json.dump(rf_meta, f, indent=2)

print('rf_best_model.joblib  saved ✅  (trained model weights)')
print('rf_model_meta.json    saved ✅  (all metrics and parameters)')
print()
print('FINAL MODEL SUMMARY')
print('='*50)
print(f'  Algorithm    : Random Forest Regressor')
print(f'  n_estimators : 100')
print(f'  max_depth    : 3')
print(f'  CV R²        : {gs.best_score_:.4f}')
print(f'  Test R²      : {r2_best:.4f}  ({r2_best*100:.1f}% variance explained)')
print(f'  Test MAE     : {mae_best:.4f} species')
print(f'  SST importance: 95.31%  |  pH importance: 4.69%')
print(f'  Residuals    : mean={residuals.mean():.2f}, std={residuals.std():.2f}')
print(f'                 range [{residuals.min():.2f}, {residuals.max():.2f}]')